# 01 · Synthetic datasets (Agrawal–Srikant / Quest model)

The ten synthetic workloads are **shipped as the parquet files used in the experiments** in `data/synthetic/`
(written in March–April 2026). All experiments use those files; nothing here overwrites them.

This notebook documents how they were generated and checks that the generator reproduces them:

* generator: `bench/agrawal_generator.py::generate_agrawal_dataset` (as used for the shipped files);
* driver: the generation loop of the shipped files (`SYNTHETIC_CONFIGS`, `seed = 42 + i`,
  `L = 2N`, `to_parquet("df_<name>.parquet")`), reproduced below;
* `MODE = "verify"` (default): regenerate every dataset up to `VERIFY_MAX_D` transactions in memory
  and compare it element-by-element with the shipped parquet;
* `MODE = "generate"`: write regenerated files to `data/synthetic_regenerated/` (T25.D5M took
  ~10 h).

**Naming.** In names such as `T20.I4.D1000k.N50k.L50k` the `L` field repeats N; the generator
was called with `L = 2N` (100k here). The `high`/`low` T15 variants differ only in
`correlation` (0.9 / 0.1). The names are kept because every result file uses them.

In [1]:
# ---- bundle configuration (same cell in every notebook) ----
import os, sys, json, time, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

BUNDLE = Path.cwd().resolve().parent          # notebooks run from <bundle>/notebooks
assert (BUNDLE / "bench" / "baseline_runners.py").exists(), f"start Jupyter in <bundle>/notebooks (cwd={Path.cwd()})"
sys.path.insert(0, str(BUNDLE / "bench"))
DATA = Path(os.environ.get("FA_DATA_DIR", BUNDLE / "data"))
RESULTS = BUNDLE / "results"
DERIVED = RESULTS / "derived"                       # regenerated statistics, tables and figures

V010 = "campaign_2026-05"                           # the v0.1.0.post1 GCE campaign
V010_DIR = RESULTS / V010
CAMPAIGN = os.environ.get("FA_CAMPAIGN", V010) # set FA_CAMPAIGN=<id> to analyse a new run
CAMPAIGN_DIR = V010_DIR if CAMPAIGN == V010 else RESULTS / "runs" / CAMPAIGN
AGG_CSV = (V010_DIR / "results_rigorous_20260523_115442.csv" if CAMPAIGN == V010
           else CAMPAIGN_DIR / f"results_rigorous_{CAMPAIGN}.csv")
LCM_CSV = V010_DIR / "lcm_declat_results_20260523_055523.csv"
COST_CSV = (DERIVED / "cost_stats.csv") if (DERIVED / "cost_stats.csv").exists() else V010_DIR / "cost_stats_20261022.csv"

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def rel(p):
    try:
        return Path(p).resolve().relative_to(BUNDLE).as_posix()
    except ValueError:
        return str(p)

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
print(f"campaign : {CAMPAIGN}\nagg csv  : {rel(AGG_CSV)}")

campaign : campaign_2026-05
agg csv  : results/campaign_2026-05/results_rigorous_20260523_115442.csv


In [2]:
import inspect
import agrawal_generator
from agrawal_generator import generate_agrawal_dataset
MODE = os.environ.get("FA_MODE", "verify")   # "verify" | "generate"
VERIFY_MAX_D = 100_000                           # verify datasets up to this many transactions
print(inspect.getsource(generate_agrawal_dataset))

def generate_agrawal_dataset(
    D: int,
    T: int,
    I: int,
    L: int,
    N: int,
    correlation: float = 0.5,
    corruption_mean: float = 0.5,
    corruption_std: float = 0.1,
    seed: int = 42,
) -> pd.DataFrame:
    """Generate synthetic transaction data using Agrawal & Srikant (1994) model.

    Parameters
    ----------
    D : int   — Number of transactions
    T : int   — Average transaction size (Poisson mean)
    I : int   — Average size of maximal potentially large itemsets
    L : int   — Number of maximal potentially large itemsets
    N : int   — Number of distinct items (item universe)
    correlation : float — Probability of picking items from previous itemset (0-1)
    corruption_mean : float — Mean of item drop probability (Normal dist)
    corruption_std  : float — Std of item drop probability
    seed : int — Random seed

    Returns
    -------
    pd.DataFrame with columns ['txn_id', 'item'] in long format.
    """
    rng = np.random.default_rng(seed)



## Parameters
`seed` is `42 + i` for the position `i` in the subset of `SYNTHETIC_CONFIGS` the loop was run over
at the time. `seed_source = verified` means regeneration with that seed reproduced the shipped
file (column `match` below is recomputed every time this notebook runs).

In [3]:
SYNTH = pd.DataFrame([
    # dataset,                    file,                                   D,       T,  I,  N,      corr, seed
    ("T12.I4.D5k.N1k.L1k",       "df_T12_I4_D5k_N1k_L1k.parquet",          5_000, 12,  4,  1_000, 0.50, 42),
    ("T5.I2.D25k.N1k.L1k",       "df_T5_I2_D25k_N1k_L1k.parquet",         25_000,  5,  2,  1_000, 0.50, 43),
    ("T50.I12.D10k.N3k.L3k",     "df_T50_I12_D10k_N3k_L3k.parquet",       10_000, 50, 12,  3_000, 0.75, 44),
    ("T15.I8.D50k.N1k.L1k.low",  "df_low_T15_I8_D50k_N1k_L1k.parquet",    50_000, 15,  8,  1_000, 0.10, 46),
    ("T30.I8.D50k.N5k.L5k",      "df_T30_I8_D50k_N5k_L5k.parquet",        50_000, 30,  8,  5_000, 0.50, 47),
    ("T3.I2.D100k.N50k.L50k",    "df_T3_I2_D100k_N50k_L50k.parquet",     100_000,  3,  2, 50_000, 0.25, 48),
    ("T10.I4.D100k.N25k.L25k",   "df_T10_I4_D100k_N25k_L25k.parquet",    100_000, 10,  4, 25_000, 0.50, 49),
    ("T20.I4.D1000k.N50k.L50k",  "df_T20_I4_D1000k_N50k_L50k.parquet", 1_000_000, 20,  4, 50_000, 0.50, 50),
    ("T30.I8.D2000k.N10k.L10k",  "df_T30_I8_D2000k_N10k_L10k.parquet", 2_000_000, 30,  8, 10_000, 0.50, 51),
    ("T15.I8.D50k.N1k.L1k.high", "df_high_T15_I8_D50k_N1k_L1k.parquet",   50_000, 15,  8,  1_000, 0.90, 42),
    ("T25.I8.D5000k.N40k.L40k",  "df_T25_I8_D5000k_N40k_L40k.parquet", 5_000_000, 25,  8, 40_000, 0.50, 43),
], columns=["dataset", "file", "D", "T", "I", "N", "corr", "seed"])
SYNTH["L"] = 2 * SYNTH["N"]
SYNTH["used_in_paper"] = SYNTH.dataset != "T50.I12.D10k.N3k.L3k"
_shipped = [DATA / "synthetic" / f for f in SYNTH.file]     # absent in the repository copy
SYNTH["sha256"] = [sha256(p) if p.exists() else "absent" for p in _shipped]
SYNTH["rows"] = [pd.read_parquet(p, columns=["txn_id"]).shape[0] if p.exists() else -1 for p in _shipped]
SYNTH[["dataset", "D", "T", "I", "N", "L", "corr", "seed", "rows", "used_in_paper", "sha256"]]

,dataset,D,T,I,N,L,corr,seed,rows,used_in_paper,sha256
0,T12.I4.D5k.N1k.L1k,5000,12,4,1000,2000,0.50,42,59852,True,01764383b97034a9a08afacd49975b45d3c8aee14175a7...
1,T5.I2.D25k.N1k.L1k,25000,5,2,1000,2000,0.50,43,124590,True,a22f1528d21c7c3dfec4948facec4625de7ffe3ec00a7d...
2,T50.I12.D10k.N3k.L3k,10000,50,12,3000,6000,0.75,44,498849,False,1e82fc688b2d6c44d7358ffc446c0759b56751b23f8cf1...
3,T15.I8.D50k.N1k.L1k.low,50000,15,8,1000,2000,0.10,46,750309,True,44636aa66ab3a7e5ec3817d3c874d6721918038559792f...
4,T30.I8.D50k.N5k.L5k,50000,30,8,5000,10000,0.50,47,1502140,True,4057777582ae24c2b4e17e2c74c1cc90663fe06940e3eb...
5,T3.I2.D100k.N50k.L50k,100000,3,2,50000,100000,0.25,48,303908,True,7a8fa6e0c4808916648dd12774855d0ca39b4ea9936f63...
6,T10.I4.D100k.N25k.L25k,100000,10,4,25000,50000,0.50,49,1000375,True,27b54793ccd6f424517ca71379aa4ee7546df8ad1c9dc1...
7,T20.I4.D1000k.N50k.L50k,1000000,20,4,50000,100000,0.50,50,20004006,True,1a27c5a965381312c992146017b545f97782305a870ea8...
8,T30.I8.D2000k.N10k.L10k,2000000,30,8,10000,20000,0.50,51,59990826,True,11ae6409de0b1d5da721cb778d1fda47a1b02ecf311c3c...
9,T15.I8.D50k.N1k.L1k.high,50000,15,8,1000,2000,0.90,42,749194,True,6f5dae8ab1667dd955d4ff07958f6140ae9f5a84a90fcc...


## Regenerate and compare

In [4]:
results = []
out_dir = DATA / "synthetic_regenerated"
for r in SYNTH.itertuples():
    if MODE == "verify" and r.D > VERIFY_MAX_D:
        results.append(dict(dataset=r.dataset, match="not checked (D > VERIFY_MAX_D)", seconds=np.nan)); continue
    t0 = time.time()
    gen = generate_agrawal_dataset(D=r.D, T=r.T, I=r.I, L=r.L, N=r.N, correlation=r.corr, seed=r.seed)
    secs = time.time() - t0
    if MODE == "generate":
        out_dir.mkdir(parents=True, exist_ok=True)
        gen.to_parquet(out_dir / r.file, index=False)
    if (DATA / "synthetic" / r.file).exists():
        shipped = pd.read_parquet(DATA / "synthetic" / r.file)
        same = (gen.shape == shipped.shape and list(gen.columns) == list(shipped.columns)
                and all((gen[c].to_numpy() == shipped[c].to_numpy()).all() for c in gen.columns))
        match = "identical" if same else "DIFFERENT"
    else:
        match = "written" if MODE == "generate" else "no shipped file"
    results.append(dict(dataset=r.dataset, match=match, seconds=round(secs, 1)))
    print(f"{r.dataset:<26} {results[-1]['match']:<10} {secs:7.1f}s")
check = SYNTH[["dataset", "seed", "used_in_paper"]].merge(pd.DataFrame(results), on="dataset")
check

T12.I4.D5k.N1k.L1k         identical      1.9s


T5.I2.D25k.N1k.L1k         identical      7.8s


T50.I12.D10k.N3k.L3k       identical     10.4s


T15.I8.D50k.N1k.L1k.low    identical     10.7s


T30.I8.D50k.N5k.L5k        identical     51.5s


T3.I2.D100k.N50k.L50k      identical    265.2s


T10.I4.D100k.N25k.L25k     identical    243.6s


T15.I8.D50k.N1k.L1k.high   identical     19.4s


,dataset,seed,used_in_paper,match,seconds
0,T12.I4.D5k.N1k.L1k,42,True,identical,1.9
1,T5.I2.D25k.N1k.L1k,43,True,identical,7.8
2,T50.I12.D10k.N3k.L3k,44,False,identical,10.4
3,T15.I8.D50k.N1k.L1k.low,46,True,identical,10.7
4,T30.I8.D50k.N5k.L5k,47,True,identical,51.5
5,T3.I2.D100k.N50k.L50k,48,True,identical,265.2
6,T10.I4.D100k.N25k.L25k,49,True,identical,243.6
7,T20.I4.D1000k.N50k.L50k,50,True,not checked (D > VERIFY_MAX_D),NaN
8,T30.I8.D2000k.N10k.L10k,51,True,not checked (D > VERIFY_MAX_D),NaN
9,T15.I8.D50k.N1k.L1k.high,42,True,identical,19.4


Parquet *bytes* of regenerated files differ from the shipped ones (the writer version is stored in
the file footer), so equality is checked on the decoded columns, not on the file hash.